In [2]:
# ============================================================
# UPSC ESSAY EVALUATION WORKFLOW
# LangGraph + Ollama + Qwen3
# ============================================================

import operator
from typing import TypedDict, Annotated

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END


# ============================================================
# 1. PYDANTIC OUTPUT SCHEMAS
# ============================================================

class EvaluationSchema(BaseModel):

    feedback: str = Field(
        description="Detailed feedback about the essay according to the evaluation criterion."
    )

    score: int = Field(
        description="Score from 0 to 10.",
        ge=0,
        le=10
    )


class FinalEvaluationSchema(BaseModel):

    feedback: str = Field(
        description="Overall evaluation of the essay with strengths, weaknesses, and practical recommendations."
    )


# ============================================================
# 2. OLLAMA MODEL
# ============================================================

model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0.1
)


# ============================================================
# 3. STRUCTURED OUTPUT MODELS
# ============================================================

structured_model = model.with_structured_output(
    EvaluationSchema
)

final_structured_model = model.with_structured_output(
    FinalEvaluationSchema
)


# ============================================================
# 4. UPSC ESSAY STATE
# ============================================================

class UPSCState(TypedDict):

    essay: str

    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str

    overall_feedback: str

    individual_scores: Annotated[
        list[int],
        operator.add
    ]

    avg_score: float


# ============================================================
# 5. LANGUAGE EVALUATOR
# ============================================================

def evaluate_language(state: UPSCState):

    print("\nEvaluating language...")

    prompt = f"""
You are a UPSC essay evaluator.

Evaluate ONLY the LANGUAGE QUALITY of the essay.

Consider:

1. Grammar
2. Vocabulary
3. Sentence construction
4. Formality
5. Word choice
6. Spelling
7. Academic writing quality
8. Readability

Do NOT primarily judge ideas or depth of analysis.

Give detailed but concise feedback.

Give a score from 0 to 10.

Essay:

{state["essay"]}
"""

    output = structured_model.invoke(prompt)

    print(f"Language Score: {output.score}/10")

    return {
        "language_feedback": output.feedback,
        "individual_scores": [output.score]
    }


# ============================================================
# 6. ANALYSIS EVALUATOR
# ============================================================

def evaluate_analysis(state: UPSCState):

    print("\nEvaluating depth of analysis...")

    prompt = f"""
You are a UPSC essay evaluator.

Evaluate ONLY the DEPTH OF ANALYSIS of the essay.

Consider:

1. Depth of arguments
2. Critical thinking
3. Multiple dimensions
4. Cause and effect
5. Advantages and disadvantages
6. Challenges
7. Solutions
8. Examples and evidence
9. Logical reasoning
10. Balance and nuance

Do NOT primarily judge grammar or writing style.

Give detailed but concise feedback.

Give a score from 0 to 10.

Essay:

{state["essay"]}
"""

    output = structured_model.invoke(prompt)

    print(f"Analysis Score: {output.score}/10")

    return {
        "analysis_feedback": output.feedback,
        "individual_scores": [output.score]
    }


# ============================================================
# 7. CLARITY OF THOUGHT EVALUATOR
# ============================================================

def evaluate_thought(state: UPSCState):

    print("\nEvaluating clarity of thought...")

    prompt = f"""
You are a UPSC essay evaluator.

Evaluate ONLY the CLARITY OF THOUGHT in the essay.

Consider:

1. Clear central argument
2. Logical flow
3. Organization of ideas
4. Coherence between paragraphs
5. Transitions
6. Relevance
7. Introduction and conclusion
8. Whether ideas are easy to follow
9. Consistency of argument

Do NOT primarily judge grammar or factual accuracy.

Give detailed but concise feedback.

Give a score from 0 to 10.

Essay:

{state["essay"]}
"""

    output = structured_model.invoke(prompt)

    print(f"Clarity Score: {output.score}/10")

    return {
        "clarity_feedback": output.feedback,
        "individual_scores": [output.score]
    }


# ============================================================
# 8. JOIN NODE
# ============================================================

def join_evaluations(state: UPSCState):

    print("\nAll three evaluations completed.")

    return {}


# ============================================================
# 9. FINAL EVALUATION
# ============================================================

def final_evaluation(state: UPSCState):

    print("\nCreating final evaluation...")

    prompt = f"""
You are a senior UPSC essay evaluator.

Create an overall evaluation of the essay using the
three evaluations below.

================ LANGUAGE =================

{state["language_feedback"]}


================ DEPTH OF ANALYSIS =================

{state["analysis_feedback"]}


================ CLARITY OF THOUGHT =================

{state["clarity_feedback"]}


Now write a concise but useful overall evaluation.

Your evaluation MUST include:

1. Strongest aspects of the essay
2. Major weaknesses
3. Specific areas the student should improve
4. Practical recommendations for improvement
5. A short concluding assessment

Do NOT give another numerical score.

Return only the overall evaluation text.
"""

    output = final_structured_model.invoke(prompt)

    overall_feedback = output.feedback

    # Calculate average score
    scores = state["individual_scores"]

    avg_score = sum(scores) / len(scores)

    print(f"\nAverage Score: {avg_score:.2f}/10")

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }


# ============================================================
# 10. CREATE LANGGRAPH
# ============================================================

graph = StateGraph(UPSCState)


# ============================================================
# 11. ADD NODES
# ============================================================

graph.add_node(
    "evaluate_language",
    evaluate_language
)

graph.add_node(
    "evaluate_analysis",
    evaluate_analysis
)

graph.add_node(
    "evaluate_thought",
    evaluate_thought
)

graph.add_node(
    "join_evaluations",
    join_evaluations
)

graph.add_node(
    "final_evaluation",
    final_evaluation
)


# ============================================================
# 12. PARALLEL EVALUATION
# ============================================================

graph.add_edge(
    START,
    "evaluate_language"
)

graph.add_edge(
    START,
    "evaluate_analysis"
)

graph.add_edge(
    START,
    "evaluate_thought"
)


# ============================================================
# 13. WAIT FOR ALL THREE EVALUATORS
# ============================================================

graph.add_edge(
    "evaluate_language",
    "join_evaluations"
)

graph.add_edge(
    "evaluate_analysis",
    "join_evaluations"
)

graph.add_edge(
    "evaluate_thought",
    "join_evaluations"
)


# ============================================================
# 14. JOIN → FINAL EVALUATION
# ============================================================

graph.add_edge(
    "join_evaluations",
    "final_evaluation"
)


# ============================================================
# 15. FINAL → END
# ============================================================

graph.add_edge(
    "final_evaluation",
    END
)


# ============================================================
# 16. COMPILE WORKFLOW
# ============================================================

workflow = graph.compile()

print("\nUPSC Essay Evaluation Workflow Ready!")


# ============================================================
# 17. ESSAY
# ============================================================

essay = """
India and AI Time

Now world change very fast because new tech call Artificial
Intel… something (AI). India also want become big in this AI
thing. If work hard, India can go top. But if no careful,
India go back.

India have many good. We have smart student, many engine-ear,
and good IT peoples. Big company like TCS, Infosys, Wipro
already use AI. Government also do program “AI for All”.
It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come,
how stop bug. In health, AI help doctor see sick early.
In school, AI help student learn good. Government office use
AI to find bad people and work fast.

But problem come also. First is many villager no have phone
or internet. So AI not help them. Second, many people lose
job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data.
Who take care? India still make data rule. If no strong rule,
AI do bad.

India must all people together – govern, school, company and
normal people. We teach AI and make sure AI not bad. Also talk
to other country and learn from them.

If India use AI good way, we become strong, help poor and make
better life. But if only rich use AI, and poor no get, then
big bad thing happen.

So, in short, AI time in India have many hope and many danger.
We must go right road. AI must help all people, not only some.
Then India grow big and world say "good job India".
"""


# ============================================================
# 18. INITIAL STATE
# ============================================================

initial_state = {
    "essay": essay,
    "individual_scores": []
}


# ============================================================
# 19. RUN WORKFLOW
# ============================================================

print("\n")
print("=" * 60)
print("STARTING UPSC EVALUATION")
print("=" * 60)

result = workflow.invoke(initial_state)


# ============================================================
# 20. DISPLAY FINAL RESULT
# ============================================================

print("\n")
print("=" * 60)
print("FINAL RESULT")
print("=" * 60)


print("\nLANGUAGE FEEDBACK:")
print(result["language_feedback"])


print("\n" + "-" * 60)

print("\nANALYSIS FEEDBACK:")
print(result["analysis_feedback"])


print("\n" + "-" * 60)

print("\nCLARITY OF THOUGHT FEEDBACK:")
print(result["clarity_feedback"])


print("\n" + "-" * 60)

print("\nOVERALL FEEDBACK:")
print(result["overall_feedback"])


print("\n" + "-" * 60)

print("\nINDIVIDUAL SCORES:")
print(result["individual_scores"])


print("\nAVERAGE SCORE:")
print(f"{result['avg_score']:.2f}/10")


print(result)


UPSC Essay Evaluation Workflow Ready!


STARTING UPSC EVALUATION

Evaluating depth of analysis...

Evaluating language...

Evaluating clarity of thought...
Language Score: 5/10
Analysis Score: 7/10
Clarity Score: 8/10

All three evaluations completed.

Creating final evaluation...

Average Score: 6.67/10


FINAL RESULT

LANGUAGE FEEDBACK:
The essay exhibits a moderate level of language quality, with some effective vocabulary and technical terms, but significant grammatical errors, inconsistent sentence structure, and informal tone. Key issues include:1. Grammar: Errors like 'Artificial Intel… something' (should be 'Artificial Intelligence'), 'If work hard' (run-on sentence), and 'AI do bad' (incorrect verb form) disrupt clarity.2. Vocabulary: While terms like 'AI for All' are appropriate, phrases like 'engine-ear' (should be 'engineer') and 'bad people' (should be 'bad actors') reduce academic sophistication.3. Sentence construction: Varied sentence lengths and complex structures (e.g